# PersonaMem Experiment: Persistent Mem0 + QA

This notebook runs a **single experiment** with:

- **Persistent on-disk memory** scoped by `experiment_id` (memory survives restarts and is isolated per experiment).
- **Data separation**: stored `user_id` is prefixed with `experiment_id` (e.g. `text_gpt-4.1.-mini_graph:123`) so records are not mixed across experiments.
- **Same config for insertion and retrieval**: graph (Neo4j) is used when possible; after **3 retries** on failure we fall back to vector-only.
- **Structured logs** under `benchmark_logs/<experiment_id>/` for filling (errors, tokens, latency) and QA (retrieved memories, Mem0 answer, correct answer, question metadata).

**Chapters:**

1. **Chapter 1: Filling the memory** — Ingest PersonaMem user bundles into Mem0.
2. **Chapter 2: Running the QA** — Run benchmark questions against memory and log results.
3. **Chapter 3: LLM-as-a-Judge** — (To be added later.)

In [1]:
import os, sys
print(sys.executable)
print(os.getpid())

/Users/viktorzozula/MemoriesTesting/PersonaMem0/.venv/bin/python
40458


In [2]:
import experiment_runner as er, mem0_full_stack as mfs, inspect, sys

print("Python:", sys.executable)
print("experiment_runner file:", er.__file__)
print("mem0_full_stack file:", mfs.__file__)
print("has close_cached_mem_clients:", hasattr(er, "close_cached_mem_clients"))
print("stage2 has skip_failed_stage1_users:",
      "skip_failed_stage1_users" in inspect.signature(er.run_stage2_qa_experiment).parameters)
print("build_full_mem0_config has enable_reranker:",
      "enable_reranker" in inspect.signature(mfs.build_full_mem0_config).parameters)

print("normalize neo4j url patch exists:",
    "_normalize_neo4j_url" in inspect.getsource(mfs))
print("graph fallback marker exists:",
    "switch_to_vector_only_failed" in inspect.getsource(er))

Python: /Users/viktorzozula/MemoriesTesting/PersonaMem0/.venv/bin/python
experiment_runner file: /Users/viktorzozula/MemoriesTesting/PersonaMem0/experiment_runner.py
mem0_full_stack file: /Users/viktorzozula/MemoriesTesting/PersonaMem0/mem0_full_stack.py
has close_cached_mem_clients: True
stage2 has skip_failed_stage1_users: True
build_full_mem0_config has enable_reranker: True
normalize neo4j url patch exists: True
graph fallback marker exists: False


## Setup

Define the **experiment ID** (used for persistent memory path and log folder) and paths to the PersonaMem bundles JSONL.

**Important:** Run this notebook from the **project root** (the `PersonaMem0` directory) so that `Path.cwd()` and relative paths (e.g. `exports/`, `benchmark_logs/`) resolve correctly.

In [3]:
import importlib
import experiment_runner as er
import mem0_full_stack as mfs

importlib.reload(mfs)
importlib.reload(er)
er.close_cached_mem_clients()

In [4]:
from pathlib import Path

# Experiment ID: memory storage and logs are scoped to this ID.
# Memory is stored under ~/.mem0/experiments/<EXPERIMENT_ID>/qdrant (persistent, on-disk).
# Logs are written to benchmark_logs/<EXPERIMENT_ID>/.
EXPERIMENT_ID = "multi_gpt-4.1.-mini_graph-img-preserved-2"

# Path to PersonaMem user bundles (chat history + benchmark rows).
# Use the multimodal 32k export for this experiment. Requires running from project root.
PROJECT_ROOT = Path.cwd()
JSONL_PATH = PROJECT_ROOT / "exports" / "personamem_benchmark_multimodal_32k_user_bundles.jsonl"

# Optional: cap users/QA for quick runs (set to None for full run).
MAX_USERS = None
MAX_QA_PER_USER = None

assert JSONL_PATH.is_file(), f"JSONL not found: {JSONL_PATH}"
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Bundles: {JSONL_PATH}")
print(f"Log dir: benchmark_logs/{EXPERIMENT_ID}/")

Experiment ID: multi_gpt-4.1.-mini_graph-img-preserved-2
Bundles: /Users/viktorzozula/MemoriesTesting/PersonaMem0/exports/personamem_benchmark_multimodal_32k_user_bundles.jsonl
Log dir: benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/


---
# Chapter 1: Filling the memory in

Load PersonaMem user bundles and add each user's chat history to Mem0. Memory is **persistent** and scoped to `EXPERIMENT_ID`. In storage, each user is stored as `EXPERIMENT_ID:user_id` so data is separated per experiment. Graph is used when possible; after 3 retries on failure we fall back to vector-only. If the LLM returns truncated or malformed JSON, a **safe-parse repair** (`mem0_safe_json`) is applied so partial memories can still be stored.

**Logs** (in `benchmark_logs/<EXPERIMENT_ID>/`):

- `stage1_fill_<timestamp>.json` — Full log: per-user wall time, input tokens, latency, success/error, add_calls, used_graph, graph_error, and config.
- `stage1_errors_<timestamp>.jsonl` — One JSON object per error line for easy parsing.

In [5]:
import importlib
import experiment_runner as er

# Refresh module state in long-lived notebooks and release local Qdrant locks.
importlib.reload(er)
er.close_cached_mem_clients()

# Use EXPERIMENT_ID from Setup; graph retries = 3 then fallback to no-graph.
log_dir = er.get_experiment_log_dir(EXPERIMENT_ID)
print(f"Log directory: {log_dir}")

log1 = er.run_stage1_fill_experiment(
    experiment_id=EXPERIMENT_ID,
    jsonl_path=JSONL_PATH,
    split="benchmark_multimodal",
    max_users=MAX_USERS,
    use_async=False,
    max_concurrent=5,
    graph_retries=3,
)

print(f"Stage 1 complete: {log1.num_users} users, {log1.total_add_calls} successful adds, {len(log1.errors)} errors")
print(f"Total wall time: {log1.total_wall_seconds:.1f}s")
if log1.total_input_tokens:
    print(f"Total input tokens (approx): {log1.total_input_tokens}")
# Whether graph (Neo4j) was used for this run (same for all users)
used_graph_s1 = log1.per_user[0].get("used_graph") if log1.per_user else None
print(f"Graph memory: {'enabled' if used_graph_s1 else 'disabled (vector-only fallback)'}")

Log directory: /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2


Stage 1 (fill memory):  27%|██▋       | 53/195 [1:42:12<2:44:02, 69.31s/user]Skipped 1 malformed graph entities from LLM tool-calls.
Error in new_retrieved_facts: Expecting value: line 1 column 10898 (char 10897)
Stage 1 (fill memory):  43%|████▎     | 84/195 [2:36:52<2:23:58, 77.83s/user]Skipped 1 malformed graph entities from LLM tool-calls.
Skipped 1 malformed graph entities from LLM tool-calls.
Stage 1 (fill memory): 100%|██████████| 195/195 [5:49:45<00:00, 107.62s/user]


Flow log: /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/flow_stage1_RUN-1_20260311_184105.log
Stage 1 log written to /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/stage1_RUN-1_fill_20260311_184105.json
Stage 1 complete: 195 users, 195 successful adds, 0 errors
Total wall time: 20985.2s
Total input tokens (approx): 6161067
Graph memory: enabled


In [6]:
# Inspect last few per-user records (wall_seconds, input_tokens, success, used_graph)
import json
for rec in log1.per_user[-5:]:
    print(json.dumps({k: rec.get(k) for k in ["user_id", "wall_seconds", "input_tokens", "success", "used_graph", "error"] if rec.get(k) is not None}, indent=2))

{
  "user_id": 985,
  "wall_seconds": 67.91038145800121,
  "input_tokens": 32069,
  "success": true,
  "used_graph": true
}
{
  "user_id": 986,
  "wall_seconds": 83.73859583301237,
  "input_tokens": 31871,
  "success": true,
  "used_graph": false
}
{
  "user_id": 989,
  "wall_seconds": 28.353130790987052,
  "input_tokens": 31823,
  "success": true,
  "used_graph": true
}
{
  "user_id": 995,
  "wall_seconds": 84.55016283295117,
  "input_tokens": 32018,
  "success": true,
  "used_graph": true
}
{
  "user_id": 998,
  "wall_seconds": 150.51346624997677,
  "input_tokens": 31636,
  "success": true,
  "used_graph": true
}


---
# Chapter 2: Running the QA

For each benchmark question we **search** memory (with graph when possible; 3 retries then fallback), then generate an **answer** with the same LLM (e.g. gpt-4.1-mini). Search uses experiment-scoped user IDs (`EXPERIMENT_ID:user_id`) so only this experiment's memories are queried. Logs include Mem0 retrieved memories, Mem0 answer, correct answer from PersonaMem, question text, and full question metadata.

**Logs** (in `benchmark_logs/<EXPERIMENT_ID>/`):

- `stage2_qa_<timestamp>.json` — Full log and summary.
- `stage2_qa_<timestamp>_stream.jsonl` — One QA record per line: `retrieved_memories`, `mem0_answer`, `correct_answer`, `user_query`, and all row fields.
- `stage2_qa_<timestamp>_answers.json` — Per-user grouping of QA results.

In [5]:
# Same experiment_id so we use the memory filled in Chapter 1.
log2 = er.run_stage2_qa_experiment(
    experiment_id=EXPERIMENT_ID,
    jsonl_path=JSONL_PATH,
    split="benchmark_multimodal",
    max_users=MAX_USERS,
    max_qa_per_user=MAX_QA_PER_USER,
    rerank=True,
    graph_retries=3,
    stream_path=None,
    resume_from_stream="benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/stage2_RUN-2_qa_20260312_003449_stream.jsonl",
    stage1_log_path=None,  # auto-picks latest Stage 1 log for filtering failed users
    skip_failed_stage1_users=True,
)

print(f"Stage 2 complete: {log2.num_qa_pairs} QA pairs, {log2.total_search_calls} searches")
print(f"Total wall time: {log2.total_wall_seconds:.1f}s")
print(f"Answer LLM calls: {log2.total_answer_llm_calls}")
if log2.total_answer_input_tokens:
    print(f"Answer input tokens: {log2.total_answer_input_tokens}")
if log2.errors:
    print(f"Errors: {len(log2.errors)}")
# Whether graph (Neo4j) was used for retrieval this run (same for all QA)
used_graph_s2 = log2.per_qa[0].get("used_graph") if log2.per_qa else None
print(f"Graph memory: {'enabled' if used_graph_s2 else 'disabled (vector-only fallback)'}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Stage 2 (QA): 100%|██████████| 5000/5000 [2:32:21<00:00,  1.83s/qa]  


Flow log: /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/flow_stage2_RUN-3_20260312_112818.log
Stage 2 log written to /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/stage2_RUN-3_qa_20260312_112818.json
Stage 2 answers written to /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/multi_gpt-4.1.-mini_graph-img-preserved-2/stage2_RUN-3_qa_20260312_112818_answers.json
Stage 2 complete: 5000 QA pairs, 5000 searches
Total wall time: 25965.2s
Answer LLM calls: 4992
Answer input tokens: 1671823
Graph memory: enabled


In [6]:
# Inspect one QA record: question, retrieved memories, mem0 answer, correct answer
import json
qa = next((q for q in log2.per_qa if q.get("retrieved_memories") and q.get("mem0_answer")), log2.per_qa[0] if log2.per_qa else None)
if qa:
    print("Question (user_query):", qa.get("user_query", "")[:200])
    print("Retrieved memories count:", len(qa.get("retrieved_memories") or []))
    print("Mem0 answer:", (qa.get("mem0_answer") or "")[:300])
    print("Correct answer (PersonaMem):", (qa.get("correct_answer") or "")[:300])
    print("Used graph:", qa.get("used_graph"))

Question (user_query): {'role': 'user', 'content': 'What are some good ways to find fresh, seasonal produce in my area without having to rely on big supermarkets?'}
Retrieved memories count: 4
Mem0 answer: To find fresh, seasonal produce locally without using big supermarkets, you might try visiting farmers' markets or local co-ops where small growers sell directly. You could also look for community-supported agriculture (CSA) programs where you subscribe to get regular boxes of fresh picks from nearb
Correct answer (PersonaMem): In Pune, you might explore your neighborhood’s early morning farmer haats or weekly subzi mandis, where local growers bring freshly harvested, seasonal produce. Building a rapport with a few trusted vendors will help you get the best picks before they sell out, much like maintaining a well-coordinat
Used graph: True


---
# Chapter 3: Running LLM-as-a-Judge

*(To be added later.)*

In [ ]:
# Placeholder: LLM-as-a-Judge evaluation will go here.
pass